# Fine-tune E5-base — 2 epoch (thử nghiệm)

Fine-tune **`intfloat/multilingual-e5-base`** với **2 epoch** để so sánh với bản **1 epoch** (`e5_base_finetuned_final`).

| | 1 epoch (đã chạy) | Notebook này (2 epoch) |
|---|---|---|
| Output model | `models/e5_base_finetuned_final/` | `models/e5_base_finetuned_2ep_final/` |
| Metrics | `metrics_e5_base.json` | `metrics_e5_base_2epochs.json` |

**Cần GPU Colab** (T4 trở lên khuyến nghị). Thời gian train ~2× so với 1 epoch.

## 1) Cài thư viện

In [ ]:
!pip -q install "sentence-transformers>=3.0.0" "transformers>=4.40.0" torch datasets pandas scikit-learn numpy tqdm

## 2) Clone repo và kiểm tra GPU

In [ ]:
import os
import shutil
import subprocess
import sys
import torch

GITHUB_REPO_URL = "https://github.com/YOUR_USER/llm_provider_benchmarking.git"  # ← sửa URL
REPO_DIR = "/content/llm_provider_benchmarking"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, REPO_DIR], check=True)

SCRIPTS_DIR = f"{REPO_DIR}/embedding_project/scripts"
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

os.chdir(REPO_DIR)
print("REPO_DIR:", REPO_DIR)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3) Cấu hình — 2 epoch

Khác bản 1 epoch: `EPOCHS = 2`, thư mục model và file metrics riêng.

In [ ]:
from pathlib import Path
from model_presets import get_preset

PRESET = get_preset("e5-base")
PROJECT_ROOT = Path(REPO_DIR) / "embedding_project"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_EVAL_DIR = PROJECT_ROOT / "outputs" / "evaluation"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_EVAL_DIR.mkdir(parents=True, exist_ok=True)

USE_GPU = torch.cuda.is_available()
EPOCHS = 2  # thử nghiệm chính
BATCH_SIZE = 8 if USE_GPU else 2
FP16 = USE_GPU
MAX_SEQ_LENGTH = PRESET.max_seq_length
LEARNING_RATE = PRESET.learning_rate
WARMUP_RATIO = PRESET.warmup_ratio

FINAL_DIR = MODELS_DIR / "e5_base_finetuned_2ep_final"
CHECKPOINT_DIR = MODELS_DIR / "e5-base-2ep"
METRICS_FILE = OUTPUT_EVAL_DIR / "metrics_e5_base_2epochs.json"
METRICS_1EP_FILE = OUTPUT_EVAL_DIR / "metrics_e5_base.json"

print("Base model:", PRESET.base_model)
print("Final dir:", FINAL_DIR)
print(f"epochs={EPOCHS} | batch={BATCH_SIZE} | fp16={FP16} | max_seq={MAX_SEQ_LENGTH} | lr={LEARNING_RATE}")

## 4) Load train / valid

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(DATA_DIR / "train_cleaned.jsonl")
valid_rows = load_jsonl(DATA_DIR / "valid_cleaned.jsonl")
print("train:", len(train_rows), "| valid:", len(valid_rows))

train_ds = Dataset.from_list([{"anchor": r["query"], "positive": r["positive"]} for r in train_rows])
valid_ds = Dataset.from_list([{"anchor": r["query"], "positive": r["positive"]} for r in valid_rows])

## 5) Fine-tune 2 epoch

Loss: `MultipleNegativesRankingLoss`. Lưu checkpoint mỗi epoch (`save_total_limit=3`).

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments, BatchSamplers

model = SentenceTransformer(PRESET.base_model)
model.max_seq_length = MAX_SEQ_LENGTH
loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    fp16=FP16,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=20,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    run_name="e5-base-vi-2epochs-colab",
    report_to=[],
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    loss=loss,
)

train_result = trainer.train()
model.save(str(FINAL_DIR))
print("Saved model:", FINAL_DIR)
print("Train summary:", train_result)

## 6) Loss theo epoch (train / valid)

In [ ]:
import pandas as pd

log_history = trainer.state.log_history
rows = []
for entry in log_history:
    if "loss" in entry and "epoch" in entry:
        rows.append({"epoch": entry["epoch"], "train_loss": entry["loss"], "step": entry.get("step")})
    if "eval_loss" in entry:
        rows.append({"epoch": entry.get("epoch"), "valid_loss": entry["eval_loss"], "step": entry.get("step")})

if rows:
    df_log = pd.DataFrame(rows)
    display(df_log.groupby("epoch", dropna=False).last())
else:
    print("Không có log_history — xem output trainer ở cell trên.")

## 7) Đánh giá trên test (chỉ fine-tuned 2 epoch)

Dùng script `evaluate_embedding_model.py` — prefix `query:` / `passage:` cho E5.

In [ ]:
!python embedding_project/scripts/evaluate_embedding_model.py \
  --preset e5-base \
  --only-finetuned \
  --finetuned-model embedding_project/models/e5_base_finetuned_2ep_final \
  --output embedding_project/outputs/evaluation/metrics_e5_base_2epochs.json \
  --no-cache

## 8) So sánh 1 epoch vs 2 epoch

In [ ]:
import json

def load_metrics(path):
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

m2 = load_metrics(METRICS_FILE)
m1 = load_metrics(METRICS_1EP_FILE)

ft2 = (m2 or {}).get("finetuned", {})
ft1 = (m1 or {}).get("finetuned", {})

metrics = ["Precision@10", "Recall@10", "MRR@10", "NDCG@10"]
rows = []
for k in metrics:
    v1 = ft1.get(k)
    v2 = ft2.get(k)
    delta = "—"
    if v1 is not None and v2 is not None and v1 != 0:
        delta = f"{(v2 - v1) / v1 * 100:+.1f}%"
    rows.append({"Metric": k, "1 epoch": v1, "2 epoch": v2, "Δ (2 vs 1)": delta})

df_cmp = pd.DataFrame(rows)
display(df_cmp)

if not ft1:
    print("Chưa có metrics_e5_base.json (1 epoch) — upload hoặc chạy eval bản 1 epoch trước.")
if ft2:
    better = sum(1 for k in metrics if ft2.get(k, 0) > ft1.get(k, 0))
    print(f"2 epoch tốt hơn 1 epoch trên {better}/{len(metrics)} metric.")

## 9) Tải model về máy (zip)

In [ ]:
import shutil
from google.colab import files

assert FINAL_DIR.is_dir(), f"Chưa có model: {FINAL_DIR}"

zip_path = shutil.make_archive("/content/e5_base_finetuned_2ep_final", "zip", root_dir=str(FINAL_DIR))
print("Zip:", zip_path)
files.download(zip_path)

## 10) (Tùy chọn) Tải metrics JSON

In [ ]:
from google.colab import files

if METRICS_FILE.is_file():
    files.download(str(METRICS_FILE))
else:
    print("Chưa có file metrics — chạy cell đánh giá trước.")